# Try/Except Error Handling

## La técnica del print() detective

In [18]:
# Bug: el total de ventas da 0 pero el CSV tiene datos
ventas = []

with open("../04_io_files/ventas.csv", encoding="utf-8",) as f:
    lineas = f.readlines()[1:]

    for linea in lineas:
        campos = linea.strip().split(",")

        # print() detective: ver que hay en campos
        print(f"DEBUG campos: {campos}") # <-- inspeccionar

        precio = campos[3]
        # print() detective: ver el tipo y valor
        print(f"DEBUG precio: {repr(precio)}, tipo: {type(precio)}") # <-- inspeccionar

        ventas.append(float(precio)) # BUG: es string, no float!

# El print revela: precio es "45.99" (string), no 45.99 (float)
# Solucion: ventas.append(float(precio))

total = sum(ventas)  # TypeError si son strings, o 0 si la lista esta vacia
print(f"DEBUG total: {total}, len(ventas): {len(ventas)}")  # <-- verificar resultado


DEBUG campos: ['2024-01-15', 'Laptop Pro', '2', '1299.99']
DEBUG precio: '1299.99', tipo: <class 'str'>
DEBUG campos: ['2024-01-15', 'Monitor 4K', '1', '449.0']
DEBUG precio: '449.0', tipo: <class 'str'>
DEBUG campos: ['2024-01-16', 'Teclado mecánico', '5', '89.99']
DEBUG precio: '89.99', tipo: <class 'str'>
DEBUG campos: ['2024-01-16', 'Ratón ergonómico', '3', '59.99']
DEBUG precio: '59.99', tipo: <class 'str'>
DEBUG total: 1898.97, len(ventas): 4


## Bloque try/except básico

In [25]:
# sin try/except - explota con datos sucios
precios = ["12.99", "45.00", "N/A", "89.50", "", "120.00"]

# Esto FALLARÁ en "N/A" y "" → ValueError
# totales = [float(p) for p in precios]

In [ ]:
# CON try/except — maneja datos sucios
totales = []
errores = []

for i, precio in enumerate(precios):
    try:
        valor = float(precio)
        totales.append(valor)
    except ValueError:
        errores.append(f"Fila {i}: '{precio}' no es un número válido")
    # except  Exception as e:
    #     print(f"Error inesperado: {type(e).__name__}: {e}")

print(f"Procesados: {len(totales)} valores")
print(f"Errores: {len(errores)} filas con problemas\n")

for error in errores:
    print(f"[AVISO] {error}")


Error inesperado: ValueError: could not convert string to float: 'N/A'
Error inesperado: ValueError: could not convert string to float: ''
Procesados: 4 valores
Errores: 0 filas con problemas



## Pattern try/except/else/finaly

In [35]:
import json

def cargar_config(ruta):
    """Cargar configuración JSON con manejo robusto de errores"""

    try:
        with open(ruta, 'r', encoding="utf-8") as f:
            config = json.load(f)
    except FileNotFoundError:
        print(f"[AVISO] {ruta} no existe. Usando configuración por defecto.")
        config = {"umbral": 100, "formato_salida": "json"}
    except json.JSONDecodeError:
        print(f"[ERROR] {ruta} tiene JSON inválido: {e}")
        config = {"umbral": 100, "formato_salida": "json"}
    else:
        # Se ejecuta SOLO si no hubo excepción
        print(f"[OK] Configuración cargada desde {ruta}")
    finally:
        # Se ejecuta SIEMPRE (haya o no error)
        print(f"  Config activa: {config}")

# Probar con archivo existente y no existente
config = cargar_config("../04_io_files/resume.json")
config2 = cargar_config("no_existe.json")



[OK] Configuración cargada desde ../04_io_files/resume.json
  Config activa: {'fecha': '2024-01-15', 'total_ventas': 3057.96, 'num_transacciones': 4, 'productos_top': ['Laptop Pro', 'Monitor 4K'], 'meta_alcanzada': True}
[AVISO] no_existe.json no existe. Usando configuración por defecto.
  Config activa: {'umbral': 100, 'formato_salida': 'json'}


## Patrón profesional: procesar con tolerancia fallos

In [ ]:
def procesar_ventas(registros):
    """Procesa ventas con tolerancia a fallos."""

    exitosos = []
    fallidos = []

    for i, reg in enumerate(registros):
        try:
            # validar campos requeridos
            if 'producto' not in reg or 'precio' not in reg:
                raise ValueError('Faltan campos requeridos')

            # convertir y calcular
            precio = float(reg['precio'])
            cantidad = int(reg.get('cantidad', 1))

            if precio < 0:
                    raise ValueError(f"Precio negativo: {precio}")

            reg['total'] = precio * cantidad
            exitosos.append(reg)

        except (ValueError, TypeError) as e:
            fallidos.append({"fila": i, "error": str(e), "registro": reg})

    return exitosos, fallidos


In [46]:
# datos con problema reales
ventas_sucias = [
    {"producto": "Laptop", "precio": "1299.99", "cantidad": "2"},
    {"producto": "Monitor", "precio": "N/A"},        # precio no numérico
    {"precio": "50.00"},                              # falta producto
    {"producto": "Teclado", "precio": "79.99", "cantidad": "3"},
    {"producto": "Cable", "precio": "-5.00"},         # precio negativo
]



In [47]:
ok, errores = procesar_ventas(ventas_sucias)

print(f"\n[OK] Procesados: {len(ok)}")
print(f"[FAIL] Con errores: {len(errores)}\n")

if errores:
    for err in errores:
        print(f"Fila {err['fila']}: {err['error']}")
else:
    print("[OK] Errorres no encontrados")


[OK] Procesados: 2
[FAIL] Con errores: 3

Fila 1: could not convert string to float: 'N/A'
Fila 2: Faltan campos requeridos
Fila 4: Precio negativo: -5.0
